# tensor-reshape-view — worked example 2: reshape after transpose — silent copy

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-reshape-view`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After a `.transpose()` or `.permute()`, the tensor is usually no longer contiguous because its strides no longer match row-major layout. Calling `.view()` on a non-contiguous tensor raises `RuntimeError`. `.reshape()` handles this silently: it attempts a no-copy view first, and if that fails, it falls back to making a contiguous copy before reshaping.

## Worked solution

**Step 1 — Create a non-contiguous tensor.**
`x.transpose(0, 1)` swaps two axes and changes the strides so the tensor is no longer contiguous. Check with `is_contiguous()`.

**Step 2 — `.view()` raises.**
Calling `.view(new_size)` on the transposed tensor raises `RuntimeError: view size is not compatible with input tensor's size and stride`. This is intentional — `.view()` is a strict no-copy guarantee.

**Step 3 — `.reshape()` succeeds.**
`.reshape(new_size)` first checks if a view is possible. If not, it calls `.contiguous()` internally and then does the view on the fresh copy. The result is correct but the `data_ptr()` will differ from the original.

**Step 4 — `data_ptr` comparison.**
Compare `y.data_ptr() == x_t.data_ptr()` to determine whether a copy occurred. After a non-contiguous transpose, reshape always copies.

In [ ]:
import torch as t

t.manual_seed(3)
x = t.randn(3, 5)       # (3, 5) contiguous
x_t = x.T               # (5, 3) — transpose, non-contiguous
print('x_t.is_contiguous():', x_t.is_contiguous())  # False

# .view() raises on non-contiguous
try:
    _ = x_t.view(15)
    print('view succeeded (unexpected)')
except RuntimeError as e:
    print('view raised RuntimeError:', str(e)[:60])

# .reshape() succeeds silently
y = x_t.reshape(15)
print('reshape shape:', y.shape)   # (15,)
print('copy happened (data_ptr differs):', y.data_ptr() != x_t.data_ptr())  # True

# Content is correct: same elements
print('values match .contiguous().view:', t.allclose(y, x_t.contiguous().view(15)))  # True